In [ ]:
from google.colab import files
uploaded_files = files.upload()

Saving Collins Scrabble Words (2019).txt to Collins Scrabble Words (2019).txt


In [ ]:
# in D:Downloads folder
file = open("Collins Scrabble Words (2019).txt", "r")
file.readline()
file.readline()
dictionary = file.readlines()

In [ ]:
def get_unique_letters(word):
  unique_letters = []
  for letter in word:
    if letter not in unique_letters:
      unique_letters.append(letter)
  unique_letters.sort()
  return unique_letters

def count_unique_letters(word):
  return len(get_unique_letters(word))

print(count_unique_letters("banana")) #3
print(count_unique_letters("super")) #5
print(count_unique_letters("desired")) #5

print(get_unique_letters("banana")) #3
print(get_unique_letters("super")) #5
print(get_unique_letters("desired")) #5

dictionary = [word.strip().lower() for word in dictionary]
dictionary = [word for word in dictionary if len(word) >= 4]
dictionary = [word for word in dictionary if count_unique_letters(word) <= 7]

seven_unique_letters_dictionary = [word for word in dictionary if count_unique_letters(word) == 7]
print(len(seven_unique_letters_dictionary))

3
5
5
['a', 'b', 'n']
['e', 'p', 'r', 's', 'u']
['d', 'e', 'i', 'r', 's']
60603


In [ ]:
from numpy import sqrt

def get_invalid_letter(valid_letters, word):
  for letter in word:
    if letter not in valid_letters:
      return letter
  return None

def get_word_score(letters_of_the_day, word):
  if len(word) < 4:
    return {"word_score": -1, "is_panagram": False}
  elif len(word) == 4:
    return {"word_score": 1, "is_panagram": False}
  word_score = len(word)
  is_panagram = False
  if count_unique_letters(word) == 7:
    word_score += 7
    is_panagram = True
  return {"word_score": word_score, "is_panagram": is_panagram}

def get_ranks(max_score):
  rank_names = ["Good Start", "Novice", "Okay", "Good", "Solid", "Nice", "Great", "Amazing", "Genius"]
  num_ranks = len(rank_names)
  rank_reqs = [0] * num_ranks
  for i in range(num_ranks):
    rank_req = (i / (num_ranks - 1)) ** 2
    rank_req *= 2.2 * max_score ** 0.62
    rank_req = round(rank_req)
    rank_reqs[i] = rank_req
  return [{"name": rank_names[i], "req": rank_reqs[i]} for i in range(num_ranks)]

def get_rank_index(ranks, score):
  for i in range(len(ranks) - 1, -1, -1):
    if score >= ranks[i]["req"]:
      return i
  return -1

In [ ]:
from random import randint, shuffle, sample

def play_game():
  word_of_the_day = seven_unique_letters_dictionary[randint(0, len(seven_unique_letters_dictionary))]
  #word_of_the_day = "wmaoryt"
  #word_of_the_day = "olactyi"
  #print("word_of_the_day: " + word_of_the_day)
  letters_of_the_day = get_unique_letters(word_of_the_day)
  shuffle(letters_of_the_day)
  #letters_of_the_day = [word_of_the_day[i] for i in range(len(word_of_the_day))]
  special_letter = letters_of_the_day[-1]

  def print_stylized_wotd(letters):
    stylized_wotd = ""
    for i in range(len(letters) - 1):
      stylized_wotd += letters[i].upper() + " "
    stylized_wotd += "*" + letters[-1].upper() + "*"
    print(f"Today's letters: ", stylized_wotd)

  print_stylized_wotd(letters_of_the_day)

  viable_words = [word for word in dictionary if get_invalid_letter(letters_of_the_day, word) is None and special_letter in word]
  maximum_score = 0
  for word in viable_words:
    maximum_score += get_word_score(letters_of_the_day, word)["word_score"]
  ranks = get_ranks(maximum_score)

  user_score = 0
  user_words = []

  while True:
    user_word = input().lower()
    if user_word == "!quit":
      rank_index = get_rank_index(ranks, user_score)
      rank_name = ranks[rank_index]["name"]
      print("Ending game\n   Panagram of the day: " + word_of_the_day + "\n   Final rank: " + rank_name + " (" + str(user_score) + ")")
      break
    if user_word == "!cheat":
      print("   Panagram of the day: " + word_of_the_day)
      samples = []
      iterations = 0
      while len(samples) < 8 and iterations < 100:
        iterations += 1
        random_viable_word = viable_words[randint(0, len(viable_words))]
        if len(random_viable_word) == 4:
          continue
        if random_viable_word in user_words:
          continue
        samples.append(random_viable_word)
      print("   Valid words: " + samples)
      continue
    if user_word == "!hints":
      print("   Number of valid words: " + str(len(viable_words)))
      print("   Maximum score possible: " + str(maximum_score))
      print("   Ranks: " + str([ranks[i]["req"] for i in range(len(ranks))]))
      continue
    if user_word == "!letters":
      print_stylized_wotd(letters_of_the_day)
      continue
    if user_word == "!shuffle":
      new_letters = letters_of_the_day[:-1]
      shuffle(new_letters)
      letters_of_the_day = new_letters + [letters_of_the_day[-1]]
      print_stylized_wotd(letters_of_the_day)
      continue
    if user_word == "!replay":
      return play_game()
    if user_word[0] == "!":
      print("   Command not recognized")
      continue

    if len(user_word) < 4:
      print("   Word too short")
      continue
    if special_letter not in user_word:
      print("   Word doesn't contain special letter: " + special_letter)
      continue
    invalid_letter = get_invalid_letter(letters_of_the_day, user_word)
    if invalid_letter is not None:
      print("   Word contains invalid letter: " + invalid_letter)
      continue
    if user_word not in dictionary:
      print("   Word not recognized")
      continue
    if user_word in user_words:
      print("   Word already tried")
      continue

    user_words.append(user_word)
    tpl = get_word_score(letters_of_the_day, user_word)
    word_score = tpl["word_score"]
    is_panagram = tpl["is_panagram"]
    user_score += word_score
    rank_index = get_rank_index(ranks, user_score)
    rank_name = ranks[rank_index]["name"]

    text = "   {uw}: {ws} points".format(uw = user_word, ws = word_score)
    if is_panagram:
      text += " (panagram!)"
    text = "{:45}".format(text)
    text += "| " + rank_name + " (" + str(user_score) + ")"
    if rank_index != len(ranks) - 1:
      text += ", " + str(ranks[rank_index + 1]["req"] - user_score) + " more to " + ranks[rank_index + 1]["name"]
    print(text)

In [ ]:
play_game()

Today's letters:  U H E O R G *T*
!hints
   Number of valid words: 184
   Maximum score possible: 926
   Ranks: [0, 2, 9, 21, 38, 59, 85, 116, 152]
otter
   otter: 5 points                           | Novice (5), 4 more to Okay
gutter
   gutter: 6 points                          | Okay (11), 10 more to Good
tugger
   tugger: 6 points                          | Okay (17), 4 more to Good
rugger
   Word doesn't contain special letter: t
greeter
   greeter: 7 points                         | Good (24), 14 more to Solid
regreet
   regreet: 7 points                         | Good (31), 7 more to Solid
greet
   greet: 5 points                           | Good (36), 2 more to Solid
retrot
   Word not recognized
trotter
   trotter: 7 points                         | Solid (43), 16 more to Nice
!quit
Ending game
   Panagram of the day: thorougher
   Final rank: Solid (43)
